In [1]:
# Import libraries

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 150)

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02_preprocessed"
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "04_predictions"
REPORTS_DIR = PROJECT_ROOT / "reports"
AUDIT_DIR = PROJECT_ROOT / "data" / "04_predictions" / "audit_logs"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("PREDICTIONS_DIR:", PREDICTIONS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("AUDIT_DIR:", AUDIT_DIR)

PROJECT_ROOT: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform
PREPROCESSED_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed
PREDICTIONS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions
REPORTS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports
AUDIT_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions\audit_logs


In [3]:
# Load outputs from prior notebooks

rag_results_path = PREPROCESSED_DIR / "sample_rag_retrieval_results.csv"
claim_predictions_path = PREDICTIONS_DIR / "claim_risk_predictions.csv"
claim_model_results_path = REPORTS_DIR / "claim_model_results.csv"

rag_results_df = pd.read_csv(rag_results_path)
claim_predictions_df = pd.read_csv(claim_predictions_path)
claim_model_results_df = pd.read_csv(claim_model_results_path)

print("RAG results shape:", rag_results_df.shape)
print("Claim predictions shape:", claim_predictions_df.shape)
print("Claim model results shape:", claim_model_results_df.shape)

display(rag_results_df.head())
display(claim_predictions_df.head())
display(claim_model_results_df.head())

RAG results shape: (5, 4)
Claim predictions shape: (171513, 7)
Claim model results shape: (2, 13)


,document_name,chunk_id,text,score
0,plan_0451_67775DE0010004.txt,plan_0451_67775DE0010004_chunk_0,Plan Name: Select Plan Basic\nPlan ID: 67775DE0010004\nIssuer: Dominion National\nState: DE\nMetal Level: Low\nPlan Type: HMO\nHSA Eligible: Not a...,0.566556
1,plan_0917_74243FL0010003.txt,plan_0917_74243FL0010003_chunk_0,Plan Name: Choice PPO Plus\nPlan ID: 74243FL0010003\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: Not ava...,0.565149
2,plan_0915_74243FL0010001.txt,plan_0915_74243FL0010001_chunk_0,Plan Name: Choice PPO Basic\nPlan ID: 74243FL0010001\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: Not av...,0.564147
3,plan_0918_74243FL0010004.txt,plan_0918_74243FL0010004_chunk_0,Plan Name: Choice PPO Preventive\nPlan ID: 74243FL0010004\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: N...,0.561745
4,plan_0916_74243FL0010002.txt,plan_0916_74243FL0010002_chunk_0,Plan Name: Choice PPO Premium\nPlan ID: 74243FL0010002\nIssuer: Dominion National\nState: FL\nMetal Level: High\nPlan Type: PPO\nHSA Eligible: Not...,0.560491


,claim_type,claim_duration_days,has_provider_id,has_diagnosis_code,actual_high_cost_claim,predicted_high_cost_claim,high_cost_claim_probability
0,outpatient,1.0,1,1,0,0,0.147173
1,outpatient,1.0,1,1,0,0,0.147173
2,outpatient,1.0,1,1,0,0,0.147173
3,outpatient,1.0,1,1,0,0,0.147173
4,outpatient,1.0,1,1,0,0,0.147173


,model,train_accuracy,test_accuracy,train_precision,test_precision,train_recall,test_recall,train_f1,test_f1,train_roc_auc,test_roc_auc,roc_auc_gap,f1_gap
0,Logistic Regression Baseline,0.926606,0.926105,0.604795,0.602973,0.817572,0.815258,0.695269,0.693227,0.906987,0.905577,0.001410,0.002041
1,Random Forest,0.937395,0.937072,0.657711,0.656890,0.810470,0.807116,0.726144,0.724296,0.907089,0.905616,0.001473,0.001848


In [4]:
# Create simple PII/PHI redaction functions
# This is a lightweight demonstration, not a full HIPAA de-identification engine.

def redact_email(text):
    return re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[REDACTED_EMAIL]",
        str(text),
    )


def redact_phone(text):
    return re.sub(
        r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b",
        "[REDACTED_PHONE]",
        str(text),
    )


def redact_ssn(text):
    return re.sub(
        r"\b\d{3}-\d{2}-\d{4}\b",
        "[REDACTED_SSN]",
        str(text),
    )


def redact_member_id(text):
    return re.sub(
        r"\b(?:member|bene|beneficiary|patient)[-_ ]?id[:\s#]*[A-Za-z0-9_-]+\b",
        "[REDACTED_MEMBER_ID]",
        str(text),
        flags=re.IGNORECASE,
    )


def redact_sensitive_text(text):
    redacted = str(text)
    redacted = redact_email(redacted)
    redacted = redact_phone(redacted)
    redacted = redact_ssn(redacted)
    redacted = redact_member_id(redacted)
    return redacted

In [5]:
# Test redaction

sample_sensitive_queries = [
    "What is the out-of-pocket maximum for member ID BENE12345?",
    "Please review claim for patient id ABC-999 and call 210-555-1234.",
    "Send this to test.user@email.com and use SSN 123-45-6789.",
]

for query in sample_sensitive_queries:
    print("Original:", query)
    print("Redacted:", redact_sensitive_text(query))
    print("---")

Original: What is the out-of-pocket maximum for member ID BENE12345?
Redacted: What is the out-of-pocket maximum for [REDACTED_MEMBER_ID]?
---
Original: Please review claim for patient id ABC-999 and call 210-555-1234.
Redacted: Please review claim for [REDACTED_MEMBER_ID] and call [REDACTED_PHONE].
---
Original: Send this to test.user@email.com and use SSN 123-45-6789.
Redacted: Send this to [REDACTED_EMAIL] and use SSN [REDACTED_SSN].
---


In [6]:
# Create helper functions for audit logging

def hash_text(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def create_audit_record(
    user_query,
    task_type,
    model_or_system,
    response_summary,
    source_documents=None,
    confidence_score=None,
    redaction_applied=True,
    human_review_required=False,
):
    redacted_query = redact_sensitive_text(user_query)

    record = {
        "audit_id": hash_text(f"{datetime.utcnow().isoformat()}_{user_query}")[:16],
        "timestamp_utc": datetime.utcnow().isoformat(),
        "task_type": task_type,
        "model_or_system": model_or_system,
        "raw_query_hash": hash_text(user_query),
        "redacted_query": redacted_query,
        "redaction_applied": redaction_applied,
        "response_summary": response_summary,
        "source_documents": source_documents if source_documents is not None else [],
        "confidence_score": confidence_score,
        "human_review_required": human_review_required,
    }

    return record

In [7]:
# Create sample RAG audit records

rag_source_documents = rag_results_df["document_name"].head(3).tolist()

rag_audit_records = [
    create_audit_record(
        user_query="What is the out-of-pocket maximum for this plan?",
        task_type="benefits_rag_retrieval",
        model_or_system="sentence-transformers/all-MiniLM-L6-v2 + FAISS",
        response_summary="Retrieved relevant CMS plan chunks for out-of-pocket maximum question.",
        source_documents=rag_source_documents,
        confidence_score=float(rag_results_df["score"].head(3).mean()),
        redaction_applied=True,
        human_review_required=False,
    ),
    create_audit_record(
        user_query="Is this plan HSA eligible for member ID ABC123?",
        task_type="benefits_rag_retrieval",
        model_or_system="sentence-transformers/all-MiniLM-L6-v2 + FAISS",
        response_summary="Retrieved relevant CMS plan chunks for HSA eligibility question.",
        source_documents=rag_source_documents,
        confidence_score=float(rag_results_df["score"].head(3).mean()),
        redaction_applied=True,
        human_review_required=False,
    ),
]

rag_audit_df = pd.DataFrame(rag_audit_records)

display(rag_audit_df)

,audit_id,timestamp_utc,task_type,model_or_system,raw_query_hash,redacted_query,redaction_applied,response_summary,source_documents,confidence_score,human_review_required
0,34136b84372fd46f,2026-05-08T05:10:53.180073,benefits_rag_retrieval,sentence-transformers/all-MiniLM-L6-v2 + FAISS,405e468bd2c794b9939c55b7b2794beae22e83edef53b43810cde711dc26a09e,What is the out-of-pocket maximum for this plan?,True,Retrieved relevant CMS plan chunks for out-of-pocket maximum question.,"[plan_0451_67775DE0010004.txt, plan_0917_74243FL0010003.txt, plan_0915_74243FL0010001.txt]",0.565284,False
1,627e04f523eb3149,2026-05-08T05:10:53.180073,benefits_rag_retrieval,sentence-transformers/all-MiniLM-L6-v2 + FAISS,6c289bfa7759be5ac0db2450983a7256dbaf4dcc08ee3f57a6b0f188d6bad379,Is this plan HSA eligible for [REDACTED_MEMBER_ID]?,True,Retrieved relevant CMS plan chunks for HSA eligibility question.,"[plan_0451_67775DE0010004.txt, plan_0917_74243FL0010003.txt, plan_0915_74243FL0010001.txt]",0.565284,False


In [8]:
# Create sample claim model audit records

claim_sample = claim_predictions_df.head(5).copy()

claim_audit_records = []

for idx, row in claim_sample.iterrows():
    probability = float(row["high_cost_claim_probability"])
    
    claim_audit_records.append(
        create_audit_record(
            user_query=f"Score claim risk for claim row {idx}",
            task_type="claim_risk_prediction",
            model_or_system="Random Forest claim_risk_model.pkl",
            response_summary=f"Predicted high-cost claim probability: {probability:.4f}",
            source_documents=["data/03_features/claim_features.csv"],
            confidence_score=probability,
            redaction_applied=True,
            human_review_required=probability >= 0.50,
        )
    )

claim_audit_df = pd.DataFrame(claim_audit_records)

display(claim_audit_df)

,audit_id,timestamp_utc,task_type,model_or_system,raw_query_hash,redacted_query,redaction_applied,response_summary,source_documents,confidence_score,human_review_required
0,27f0d7fda919ff94,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,847186cd2f928773d6e9c80dfb7c629aa03d61048583ad4bfc73fe2493782482,Score claim risk for claim row 0,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
1,a0fec81e9186b39c,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,9e6d82c1f0b9e5eb35dbf045fa31cbf245d7b28f60167b3f2cb64b9d6574d2fb,Score claim risk for claim row 1,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
2,e540fbd7b844b0e4,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,c8b8735530f075c156671255d70a5466da7dac5b7f4edb6eabb55b23558d199a,Score claim risk for claim row 2,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
3,8473c82817e92f39,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,331cc3d055e8ed3e68edaba01d5a8ce2ad8394716ee4fcda83406dc91c11f92f,Score claim risk for claim row 3,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
4,48194d340b9e7f3b,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,fa887b1986c224f72b98afb7c526a3f2c7f5558d9d820d616afc69405197e102,Score claim risk for claim row 4,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False


In [9]:
# Combine audit logs

audit_log_df = pd.concat(
    [
        rag_audit_df,
        claim_audit_df,
    ],
    ignore_index=True,
)

print("Audit log shape:", audit_log_df.shape)
display(audit_log_df)

Audit log shape: (7, 11)


,audit_id,timestamp_utc,task_type,model_or_system,raw_query_hash,redacted_query,redaction_applied,response_summary,source_documents,confidence_score,human_review_required
0,34136b84372fd46f,2026-05-08T05:10:53.180073,benefits_rag_retrieval,sentence-transformers/all-MiniLM-L6-v2 + FAISS,405e468bd2c794b9939c55b7b2794beae22e83edef53b43810cde711dc26a09e,What is the out-of-pocket maximum for this plan?,True,Retrieved relevant CMS plan chunks for out-of-pocket maximum question.,"[plan_0451_67775DE0010004.txt, plan_0917_74243FL0010003.txt, plan_0915_74243FL0010001.txt]",0.565284,False
1,627e04f523eb3149,2026-05-08T05:10:53.180073,benefits_rag_retrieval,sentence-transformers/all-MiniLM-L6-v2 + FAISS,6c289bfa7759be5ac0db2450983a7256dbaf4dcc08ee3f57a6b0f188d6bad379,Is this plan HSA eligible for [REDACTED_MEMBER_ID]?,True,Retrieved relevant CMS plan chunks for HSA eligibility question.,"[plan_0451_67775DE0010004.txt, plan_0917_74243FL0010003.txt, plan_0915_74243FL0010001.txt]",0.565284,False
2,27f0d7fda919ff94,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,847186cd2f928773d6e9c80dfb7c629aa03d61048583ad4bfc73fe2493782482,Score claim risk for claim row 0,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
3,a0fec81e9186b39c,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,9e6d82c1f0b9e5eb35dbf045fa31cbf245d7b28f60167b3f2cb64b9d6574d2fb,Score claim risk for claim row 1,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
4,e540fbd7b844b0e4,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,c8b8735530f075c156671255d70a5466da7dac5b7f4edb6eabb55b23558d199a,Score claim risk for claim row 2,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
5,8473c82817e92f39,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,331cc3d055e8ed3e68edaba01d5a8ce2ad8394716ee4fcda83406dc91c11f92f,Score claim risk for claim row 3,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False
6,48194d340b9e7f3b,2026-05-08T05:10:53.513556,claim_risk_prediction,Random Forest claim_risk_model.pkl,fa887b1986c224f72b98afb7c526a3f2c7f5558d9d820d616afc69405197e102,Score claim risk for claim row 4,True,Predicted high-cost claim probability: 0.1472,[data/03_features/claim_features.csv],0.147173,False


In [10]:
# Save audit logs

audit_csv_path = AUDIT_DIR / "ai_audit_log.csv"
audit_json_path = AUDIT_DIR / "ai_audit_log.json"

audit_log_df.to_csv(audit_csv_path, index=False)
audit_log_df.to_json(audit_json_path, orient="records", indent=2)

print("Saved audit CSV:", audit_csv_path)
print("Saved audit JSON:", audit_json_path)

Saved audit CSV: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions\audit_logs\ai_audit_log.csv
Saved audit JSON: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions\audit_logs\ai_audit_log.json


In [11]:
# Create governance metrics

governance_metrics = {
    "total_audit_events": len(audit_log_df),
    "rag_events": int((audit_log_df["task_type"] == "benefits_rag_retrieval").sum()),
    "claim_prediction_events": int((audit_log_df["task_type"] == "claim_risk_prediction").sum()),
    "redaction_applied_rate": float(audit_log_df["redaction_applied"].mean()),
    "human_review_required_count": int(audit_log_df["human_review_required"].sum()),
    "average_confidence_score": float(audit_log_df["confidence_score"].mean()),
}

governance_metrics_df = pd.DataFrame([governance_metrics])

display(governance_metrics_df)

,total_audit_events,rag_events,claim_prediction_events,redaction_applied_rate,human_review_required_count,average_confidence_score
0,7,2,5,1.0,0,0.266633


In [12]:
# Save governance metrics

governance_metrics_path = REPORTS_DIR / "governance_metrics.csv"

governance_metrics_df.to_csv(governance_metrics_path, index=False)

print("Saved governance metrics:", governance_metrics_path)

Saved governance metrics: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\governance_metrics.csv


In [13]:
# Create SOC 2-style control mapping

soc2_control_mapping = """
# SOC 2-Style Control Mapping

## Purpose

This document maps BenefitsAI project features to SOC 2-style security, availability, confidentiality, and processing integrity practices.

## Control Areas

### 1. Security

Implemented or planned controls:

1. Audit logs for AI and ML system activity
2. Query hashing to avoid storing raw sensitive inputs
3. Redaction of emails, phone numbers, SSNs, and member identifiers
4. Planned role-based access control
5. Planned secure API authentication

### 2. Confidentiality

Implemented or planned controls:

1. Public and synthetic datasets only
2. No real protected health information used
3. Redacted user queries stored in audit logs
4. Source document tracking for RAG responses
5. Planned secrets management for API keys

### 3. Processing Integrity

Implemented or planned controls:

1. Model card documenting model purpose, target, features, and limitations
2. Target leakage prevention by excluding payment-derived features
3. Saved model evaluation results
4. Saved prediction outputs for reproducibility
5. Human review flag for high-risk predictions

### 4. Availability

Implemented or planned controls:

1. Saved model artifact for repeatable inference
2. Saved FAISS vector index for retrieval
3. Planned FastAPI service
4. Planned Streamlit dashboard
5. Planned Docker deployment

## Evidence Files

1. reports/model_card.md
2. reports/claim_model_results.csv
3. data/04_predictions/claim_risk_predictions.csv
4. data/04_predictions/audit_logs/ai_audit_log.csv
5. data/04_predictions/audit_logs/ai_audit_log.json
6. reports/governance_metrics.csv
7. vector_db/cms_plan_rag.index
8. vector_db/cms_plan_chunks_metadata.pkl
"""

In [14]:
# Save SOC 2-style control mapping

soc2_path = REPORTS_DIR / "soc2_control_mapping.md"

soc2_path.write_text(soc2_control_mapping.strip(), encoding="utf-8")

print("Saved SOC 2-style control mapping:", soc2_path)

Saved SOC 2-style control mapping: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\soc2_control_mapping.md


In [15]:
# Create HIPAA privacy notes

hipaa_privacy_notes = """
# HIPAA Privacy Notes

## Purpose

This document explains how the BenefitsAI project handles healthcare and insurance data in a privacy-aware way.

## Data Used

BenefitsAI uses public and synthetic datasets:

1. CMS Exchange Public Use Files
2. CMS DE-SynPUF synthetic Medicare claims data

The project does not use real protected health information.

## Privacy Practices Demonstrated

### 1. Public and Synthetic Data

The project avoids real patient-level PHI and uses synthetic Medicare claims data for claims modeling.

### 2. Redaction

The project includes lightweight redaction functions for:

1. Email addresses
2. Phone numbers
3. Social Security numbers
4. Member or patient identifiers

### 3. Audit Logging

The project logs:

1. Timestamp
2. Task type
3. Model or system used
4. Redacted query
5. Source documents
6. Confidence score
7. Human review flag

### 4. Human Review

High-risk claim predictions are flagged for human review rather than automatic decisioning.

## Limitations

1. Regex redaction is not a complete HIPAA de-identification system.
2. Production systems should use stronger PHI detection tools.
3. Production systems should include access control, encryption, secrets management, and monitoring.
4. AI outputs should not be used for final healthcare or claim decisions without human review.

## Intended Use

This project demonstrates privacy-aware AI/ML engineering practices for healthcare and benefits administration workflows.
"""

In [16]:
# Save HIPAA privacy notes

hipaa_path = REPORTS_DIR / "hipaa_privacy_notes.md"

hipaa_path.write_text(hipaa_privacy_notes.strip(), encoding="utf-8")

print("Saved HIPAA privacy notes:", hipaa_path)

Saved HIPAA privacy notes: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\hipaa_privacy_notes.md


In [17]:
# Create risk assessment

risk_assessment = """
# BenefitsAI Risk Assessment

## Purpose

This document identifies key risks in the BenefitsAI system and describes mitigation strategies.

## Key Risks

### 1. Incorrect RAG Retrieval

Risk:
The retrieval system may return irrelevant plan records.

Mitigation:
Use source-backed retrieval, confidence scores, and human review for sensitive use cases.

### 2. Hallucinated LLM Answers

Risk:
If an LLM is added, it may generate unsupported answers.

Mitigation:
Use retrieved context only, cite sources, and instruct the model to say when information is unavailable.

### 3. Target Leakage

Risk:
The model could use features that directly define the target.

Mitigation:
Payment amount fields were excluded from the claim risk model because the target is based on payment amount.

### 4. Overfitting

Risk:
The model may perform well on training data but poorly on new data.

Mitigation:
Train/test split, test metrics, overfitting gap checks, and controlled Random Forest depth.

### 5. Privacy Exposure

Risk:
User queries could contain sensitive identifiers.

Mitigation:
Redaction functions, query hashing, and audit logs that avoid storing raw sensitive text.

### 6. Automation Bias

Risk:
Users may overtrust model predictions.

Mitigation:
Use the model for triage and prioritization only. Human review remains required for high-risk cases.

## Overall Risk Rating

Medium.

The project uses public and synthetic data and includes basic governance controls, but production deployment would require stronger security, access control, monitoring, and compliance review.
"""

In [18]:
# Save risk assessment

risk_path = REPORTS_DIR / "risk_assessment.md"

risk_path.write_text(risk_assessment.strip(), encoding="utf-8")

print("Saved risk assessment:", risk_path)

Saved risk assessment: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\risk_assessment.md


In [19]:
# Final notebook summary

print("Notebook 5 complete.")

print("\nFiles created:")
print("1. data/04_predictions/audit_logs/ai_audit_log.csv")
print("2. data/04_predictions/audit_logs/ai_audit_log.json")
print("3. reports/governance_metrics.csv")
print("4. reports/soc2_control_mapping.md")
print("5. reports/hipaa_privacy_notes.md")
print("6. reports/risk_assessment.md")

Notebook 5 complete.

Files created:
1. data/04_predictions/audit_logs/ai_audit_log.csv
2. data/04_predictions/audit_logs/ai_audit_log.json
3. reports/governance_metrics.csv
4. reports/soc2_control_mapping.md
5. reports/hipaa_privacy_notes.md
6. reports/risk_assessment.md
